# 04 · Graph-Aware Causal Model — Exposure-Mapping DML

This notebook implements and evaluates the **graph-aware causal framework** for estimating downstream spillover effects of retail promotions across related products.

**Key idea:** Instead of treating spillover features as static confounders (B4), we model treatment as a bivariate vector $(T_i, e^{sub}_i, e^{comp}_i)$ where the exposure variables $e_i = \frac{\sum_j w_{ij} T_j}{\sum_j w_{ij}}$ use the graph edge weights (PPMI for complements, cosine similarity for substitutes).

**Three causal effects estimated jointly:**

| Effect | Formula | Interpretation |
|--------|---------|----------------|
| Direct | $\beta_1$ | Own-promotion lift |
| Substitute spillover | $\beta_2$ | Cannibalization when substitutes are promoted |
| Complement spillover | $\beta_3$ | Attachment lift when complements are promoted |

**Pipeline:**
1. Load panel, graphs, and embeddings
2. Compute weighted neighborhood exposures
3. Fit exposure-mapping DML
4. Compare against baselines
5. CATE decomposition by department
6. Sanity checks (directional, covariate balance, placebo)
7. Semi-synthetic evaluation

In [ ]:
import sys
from pathlib import Path

repo_root = Path.cwd().parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import numpy as np
import pandas as pd
import scipy.sparse as sp
import yaml
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="muted", font_scale=1.05)
PALETTE = sns.color_palette("muted")

cfg = yaml.safe_load((repo_root / "configs" / "causal_config.yaml").read_text())
print("Config loaded ✓")

## 1 — Load data

In [ ]:
panel = pd.read_parquet(repo_root / cfg["data"]["panel_path"])
A_sub = sp.load_npz(str(repo_root / cfg["data"]["a_sub_path"]))
A_comp = sp.load_npz(str(repo_root / cfg["data"]["a_comp_path"]))
pid_idx = np.load(repo_root / cfg["data"]["pid_idx_path"])
embeddings = np.load(repo_root / cfg["data"]["embeddings_path"])

print(f"Panel: {panel.shape}")
print(f"A_sub:  {A_sub.shape}, nnz={A_sub.nnz:,}")
print(f"A_comp: {A_comp.shape}, nnz={A_comp.nnz:,}")
print(f"Embeddings: {embeddings.shape}")

## 2 — Temporal split and exposure computation

In [ ]:
from src.data.splits import create_temporal_split
from src.causal.exposure import add_exposures

splits = create_temporal_split(
    panel,
    train_weeks=tuple(cfg["split"]["train_weeks"]),
    val_weeks=tuple(cfg["split"]["val_weeks"]),
    test_weeks=tuple(cfg["split"]["test_weeks"]),
)

train = splits["train"]
print(f"Train: {len(train):,} rows, weeks {train['WEEK_NO'].min()}-{train['WEEK_NO'].max()}")

# Notebook sample — full run is in experiments/graph_models/run_graph_dml.py
NB_SAMPLE_N = 1_000_000
train_sample = train.sample(n=NB_SAMPLE_N, random_state=cfg["seed"])
print(f"Notebook sample: {len(train_sample):,} rows")

# Preview exposure computation on the sample
train_exp = add_exposures(
    train_sample, A_sub, pid_idx, A_comp, pid_idx,
    treatment_col=cfg["treatment"]["primary"],
    weight_type="edge_weight",
)

exp_cols = [c for c in train_exp.columns if "exposure" in c]
print(f"\nExposure columns: {exp_cols}")
train_exp[["STORE_ID", "PRODUCT_ID", "WEEK_NO", "promo_any"] + exp_cols].describe()

## 3 — Fit the graph-aware DML

Run the exposure-mapping DML with edge-weighted exposures. This jointly estimates the direct effect, substitute spillover, and complement spillover.

In [ ]:
from src.causal.graph_dml import run_graph_dml, graph_result_to_dataframe

result = run_graph_dml(
    panel=train_sample,
    A_sub=A_sub, sub_pid_idx=pid_idx,
    A_comp=A_comp, comp_pid_idx=pid_idx,
    outcome_col=cfg["outcome"]["primary"],
    treatment_col=cfg["treatment"]["primary"],
    weight_type="edge_weight",
    n_folds=cfg["dml"]["n_folds"],
    embeddings=embeddings,
    emb_pid_idx=pid_idx,
    confidence_level=cfg["dml"]["confidence_level"],
    seed=cfg["seed"],
    log_transform=cfg["dml"]["log_transform"],
    min_demand_filter=cfg["dml"]["min_demand_filter"],
    include_discount_rate=cfg["dml"].get("include_discount_rate", False),
)

graph_result_to_dataframe(result)

In [ ]:
filtered_idx = result.details.get("filtered_index")
train_model = train_sample.loc[filtered_idx].copy() if filtered_idx is not None else train_sample.copy()

diag = pd.DataFrame({
    "metric": [
        "rows_before_filter",
        "rows_after_filter",
        "zero_share_after_filter",
        "promo_share_after_filter",
        "include_discount_rate",
        "n_folds",
    ],
    "value": [
        len(train_sample),
        len(train_model),
        float((train_model[cfg["outcome"]["primary"]] == 0).mean()),
        float(train_model[cfg["treatment"]["primary"]].mean()),
        bool(cfg["dml"].get("include_discount_rate", False)),
        int(cfg["dml"]["n_folds"]),
    ],
})
display(diag)

treated_control = (
    train_model
    .assign(log_units=np.log1p(train_model[cfg["outcome"]["primary"]]))
    .groupby(cfg["treatment"]["primary"])[[cfg["outcome"]["primary"], "log_units"]]
    .mean()
    .rename(index={0: "control", 1: "treated"})
)
display(treated_control)

## 4 — Compare against baselines

Load the baseline results (from NB 03 or the CSV) and plot all effects side by side.

In [ ]:
from src.causal.interpretability import compare_effects

# Graph model effects
effects = compare_effects(result)
display(effects)

# Load baseline ATEs for comparison
baseline_csv = repo_root / cfg["output"]["tables_dir"] / "baseline_results.csv"
if baseline_csv.exists():
    baselines = pd.read_csv(baseline_csv)
else:
    from src.models.baselines import run_all_baselines, results_to_dataframe
    bl = run_all_baselines(
        panel=train, outcome_col=cfg["outcome"]["primary"],
        treatment_col=cfg["treatment"]["primary"],
        n_folds=cfg["dml"]["n_folds"],
        embeddings=embeddings, emb_pid_idx=pid_idx,
        confidence_level=cfg["dml"]["confidence_level"], seed=cfg["seed"],
        include_discount_rate=cfg["dml"].get("include_discount_rate", False),
    )
    baselines = results_to_dataframe(bl)

# Build comparison chart
graph_df = graph_result_to_dataframe(result)
graph_direct = graph_df[graph_df["effect"] == "direct"].iloc[0]

# Note: graph model ATE is on log(1+Y) scale if log_transform is enabled;
# baselines use the same covariate spec but remain on the level scale unless
# they are re-fit with the same outcome transform.
log_note = " [log(1+Y)]" if cfg["dml"].get("log_transform", False) else ""

compare = pd.concat([
    baselines[["model", "ATE", "CI_lower", "CI_upper"]].assign(source="baseline"),
    pd.DataFrame([{
        "model": f"graph_dml_edge_weight{log_note}",
        "ATE": graph_direct["ATE"],
        "CI_lower": graph_direct["CI_lower"],
        "CI_upper": graph_direct["CI_upper"],
        "source": "graph",
    }]),
], ignore_index=True)

fig, ax = plt.subplots(figsize=(9, 4))
colors = [PALETTE[0]] * len(baselines) + [PALETTE[3]]
y_pos = range(len(compare))
ax.barh(y_pos, compare["ATE"], xerr=[
    compare["ATE"] - compare["CI_lower"],
    compare["CI_upper"] - compare["ATE"],
], color=colors, capsize=4, edgecolor="black", linewidth=0.5)
ax.set_yticks(y_pos)
ax.set_yticklabels(compare["model"])
ax.axvline(0, color="grey", ls="--", lw=0.8)
ax.set_xlabel("Direct ATE")
ax.set_title("Direct Effect: Baselines vs Graph-Aware DML")
plt.tight_layout()
plt.savefig(repo_root / cfg["output"]["figures_dir"] / "baselines_vs_graph_direct.png", dpi=150)
plt.show()

## 5 — Spillover effect visualisation

The graph-aware model uniquely provides **spillover ATEs**. Visualise the three effects and their confidence intervals.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.5))

effect_names = effects["effect"].values
ates = effects["ATE"].values
ci_lo = effects["CI_lower"].values
ci_hi = effects["CI_upper"].values
colors_eff = [PALETTE[3], PALETTE[0], PALETTE[2]]

y_pos = range(len(effect_names))
ax.barh(y_pos, ates, xerr=[ates - ci_lo, ci_hi - ates],
        color=colors_eff, capsize=5, edgecolor="black", linewidth=0.5)
ax.set_yticks(y_pos)
ax.set_yticklabels(effect_names)
ax.axvline(0, color="grey", ls="--", lw=0.8)
effect_scale = "ATE (log(1+units sold))" if cfg["dml"].get("log_transform", False) else "ATE (units sold)"
ax.set_xlabel(effect_scale)
ax.set_title("Graph-Aware DML: Direct + Spillover Effects")
plt.tight_layout()
plt.savefig(repo_root / cfg["output"]["figures_dir"] / "spillover_effects.png", dpi=150)
plt.show()

## 6 — CATE decomposition by department

Break down the conditional average treatment effects by product department to identify which categories experience the strongest cannibalization and cross-selling.

In [ ]:
from src.causal.interpretability import cate_by_group, top_spillover_products

cate_dept = cate_by_group(train_sample, result, group_col="DEPARTMENT")
display(cate_dept)

# Heatmap of mean CATEs by department
fig, ax = plt.subplots(figsize=(10, max(4, len(cate_dept) * 0.4)))
heatmap_data = cate_dept[["direct_mean", "sub_spill_mean", "comp_spill_mean"]]
heatmap_data.columns = ["Direct", "Substitute\nspillover", "Complement\nspillover"]
sns.heatmap(heatmap_data, annot=True, fmt=".3f", center=0,
            cmap="RdYlGn", ax=ax, linewidths=0.5)
ax.set_title("Mean CATE by Department")
plt.tight_layout()
plt.savefig(repo_root / cfg["output"]["figures_dir"] / "cate_by_department.png", dpi=150)
plt.show()

### 6c — Cannibalization heterogeneity

The aggregate substitute spillover ATE is near zero (+0.002, CI crossing zero), but this masks meaningful **heterogeneity across departments**. Cannibalization is not a uniform phenomenon — it concentrates in categories where products are genuinely interchangeable at the item level.

Below we surface departments where the mean CATE for substitute spillover is **negative**, i.e. where promoting substitutes measurably draws demand away from the focal product.

In [ ]:
cannibal_depts = (
    cate_dept[cate_dept["sub_spill_mean"] < 0]
    [["sub_spill_mean", "sub_spill_std", "direct_mean", "n_obs"]]
    .sort_values("sub_spill_mean")
)
print(f"{len(cannibal_depts)} of {len(cate_dept)} departments show negative substitute spillover (cannibalization):")
display(cannibal_depts)

# Plot sub spillover by department, highlighting negative (cannibalization) departments
fig, ax = plt.subplots(figsize=(10, max(4, len(cate_dept) * 0.35)))
colors_sub = ["#d62728" if v < 0 else "#aec7e8" for v in cate_dept["sub_spill_mean"]]
ax.barh(
    cate_dept.index,
    cate_dept["sub_spill_mean"],
    xerr=cate_dept["sub_spill_std"],
    color=colors_sub, capsize=3, edgecolor="black", linewidth=0.4,
)
ax.axvline(0, color="grey", ls="--", lw=0.8)
ax.set_xlabel("Mean substitute spillover CATE (log(1+units))")
ax.set_title("Substitute Spillover by Department\n(red = cannibalization)")
plt.tight_layout()
plt.savefig(repo_root / cfg["output"]["figures_dir"] / "cannibalization_by_dept.png", dpi=150)
plt.show()

## 7 — Top spillover-affected products

Identify products with the most extreme substitute and complement spillover effects — these are actionable insights for promotion planning.

In [ ]:
print("Top 15 products by substitute spillover (cannibalization):")
top_sub = top_spillover_products(train_sample, result, effect_type="sub_spillover", top_k=15)
display(top_sub)

print("\nTop 15 products by complement spillover (attachment lift):")
top_comp = top_spillover_products(train_sample, result, effect_type="comp_spillover", top_k=15)
display(top_comp)

## 8 — Sanity checks

### 8a — Directional check
Verify the estimated effects have the expected economic signs:
- Direct effect > 0 (promotions increase own demand)
- Substitute spillover < 0 (cannibalization)
- Complement spillover > 0 (attachment buying)

> **Note:** The aggregate substitute spillover ATE may appear near zero or positive if cannibalization is concentrated in a subset of categories (heterogeneous effects washing out in the population average). See section 6c for the department-level breakdown.

In [ ]:
from src.evaluation.metrics import directional_check, covariate_balance

dir_check = directional_check(result)
display(dir_check)

### 8b — Covariate balance

In [ ]:
balance = covariate_balance(
    train,
    treatment_col=cfg["treatment"]["primary"],
    covariate_cols=["WEEK_NO", "STORE_ID"],
)
display(balance)

## 9 — Semi-synthetic evaluation

Use the real graph structure and covariates but inject *known* treatment effects via the DGP:

$$Y_i = \beta_0 + \beta_1 T_i + \beta_2 e^{sub}_i + \beta_3 e^{comp}_i + X_i\gamma + \varepsilon_i$$

This lets us measure bias, RMSE, and CI coverage against ground truth.

In [ ]:
from src.evaluation.synthetic_dgp import SyntheticDGPConfig, generate_synthetic_outcome
from src.evaluation.metrics import full_synthetic_evaluation

# Notebook uses a smaller sample + fewer sims for speed.
# Full 50-sim study runs in experiments/graph_models/run_graph_dml.py.
NB_SIM_SAMPLE_N = 200_000
NB_N_SIMS = 5

train_sim = train.sample(n=NB_SIM_SAMPLE_N, random_state=cfg["seed"])
print(f"Sim sample: {len(train_sim):,} rows  |  sims: {NB_N_SIMS}")

rng = np.random.default_rng(cfg["seed"])
sim_rows = []

for sim_id in range(NB_N_SIMS):
    dgp_cfg = SyntheticDGPConfig(
        true_direct_effect=cfg["synthetic"]["true_direct_effect"],
        true_sub_spillover=cfg["synthetic"]["true_sub_spillover"],
        true_comp_spillover=cfg["synthetic"]["true_comp_spillover"],
        noise_std=cfg["synthetic"]["noise_std"],
        confounding_strength=cfg["synthetic"]["confounding_strength"],
        baseline_demand=cfg["synthetic"].get("baseline_demand", 1.5),
        log_space=cfg["dml"]["log_transform"],
        seed=int(rng.integers(0, 2**31)),
    )

    sim_panel, true_effects = generate_synthetic_outcome(
        panel=train_sim, A_sub=A_sub, sub_pid_idx=pid_idx,
        A_comp=A_comp, comp_pid_idx=pid_idx,
        config=dgp_cfg,
    )

    sim_result = run_graph_dml(
        panel=sim_panel,
        A_sub=A_sub, sub_pid_idx=pid_idx,
        A_comp=A_comp, comp_pid_idx=pid_idx,
        outcome_col="y_synthetic",
        treatment_col=cfg["treatment"]["primary"],
        weight_type="edge_weight",
        n_folds=cfg["dml"]["n_folds"],
        embeddings=embeddings, emb_pid_idx=pid_idx,
        confidence_level=cfg["dml"]["confidence_level"],
        seed=cfg["seed"],
        log_transform=cfg["dml"]["log_transform"],
        min_demand_filter=cfg["dml"]["min_demand_filter"],
        include_discount_rate=cfg["dml"].get("include_discount_rate", False),
    )

    gdf = graph_result_to_dataframe(sim_result)
    for _, row in gdf.iterrows():
        sim_rows.append({
            "sim_id": sim_id,
            "effect": row["effect"],
            "true_value": true_effects[row["effect"]],
            "estimate": row["ATE"],
            "ci_lower": row["CI_lower"],
            "ci_upper": row["CI_upper"],
        })

    print(f"  sim {sim_id+1}/{NB_N_SIMS} done")

sim_results = pd.DataFrame(sim_rows)
eval_summary = full_synthetic_evaluation(sim_results)
display(eval_summary)


## 10 — Pricing effect: discount depth as treatment

Re-run the graph-aware DML with **`discount_rate`** (continuous, 0–1) as the primary treatment instead of binary `promo_any`. This answers a different but complementary question:

> *How does the depth of a price discount — and the depth of discounts on related products — affect demand?*

The neighborhood exposures now become **average discount rate among graph-neighbors**, so the three estimated effects are:

| Effect | Interpretation |
|--------|---------------|
| Direct | Own discount depth → own demand (expected: **positive**) |
| Sub spillover | Neighbor substitute discount → own demand (expected: **negative** — price competition) |
| Comp spillover | Neighbor complement discount → own demand (expected: **positive** — basket attachment) |

Note: `include_discount_rate=False` since discount rate is the treatment here, not a confounder.

In [ ]:
discount_result = run_graph_dml(
    panel=train_sample,
    A_sub=A_sub, sub_pid_idx=pid_idx,
    A_comp=A_comp, comp_pid_idx=pid_idx,
    outcome_col=cfg["outcome"]["primary"],
    treatment_col=cfg["treatment"]["continuous"],   # "discount_rate"
    weight_type="edge_weight",
    n_folds=cfg["dml"]["n_folds"],
    embeddings=embeddings,
    emb_pid_idx=pid_idx,
    confidence_level=cfg["dml"]["confidence_level"],
    seed=cfg["seed"],
    log_transform=cfg["dml"]["log_transform"],
    min_demand_filter=cfg["dml"]["min_demand_filter"],
    include_discount_rate=False,   # discount_rate is T, not W
)

discount_df = graph_result_to_dataframe(discount_result)
display(discount_df)

# Plot three effects for the discount model
discount_effects = compare_effects(discount_result)
fig, ax = plt.subplots(figsize=(7, 3.5))
d_names = discount_effects["effect"].values
d_ates = discount_effects["ATE"].values
d_lo = discount_effects["CI_lower"].values
d_hi = discount_effects["CI_upper"].values
ax.barh(range(len(d_names)), d_ates,
        xerr=[d_ates - d_lo, d_hi - d_ates],
        color=colors_eff, capsize=5, edgecolor="black", linewidth=0.5)
ax.set_yticks(range(len(d_names)))
ax.set_yticklabels(d_names)
ax.axvline(0, color="grey", ls="--", lw=0.8)
ax.set_xlabel("ATE (log(1+units sold) per unit discount rate)")
ax.set_title("Graph-Aware DML: Discount Depth Treatment Effects")
plt.tight_layout()
plt.savefig(repo_root / cfg["output"]["figures_dir"] / "discount_spillover_effects.png", dpi=150)
plt.show()

## 11 — Save all results

In [ ]:
out_dir = repo_root / cfg["output"]["tables_dir"]
out_dir.mkdir(parents=True, exist_ok=True)

graph_result_to_dataframe(result).to_csv(out_dir / "graph_model_results.csv", index=False)
effects.to_csv(out_dir / "effect_comparison.csv", index=False)
cate_dept.to_csv(out_dir / "cate_by_department.csv")
eval_summary.to_csv(out_dir / "synthetic_evaluation.csv", index=False)
discount_df.to_csv(out_dir / "discount_results.csv", index=False)

print(f"All results saved to {out_dir}")
print("\nFinal effect comparison (promo_any):")
display(effects)
print("\nDiscount depth effects:")
display(discount_df)